# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Louai-Mohamed/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Two audits in one notebook. First, two findings from the FlyRank research paper, read the way I would want
someone to read my own work: where does the number come from, and does the validation design support the
claim? Then the same lens turned on my Week-5 model — **the same model, the same features, the same label,
scored under the split I used and under the split I should have used.**

The paper is a fair target because it already discloses its own limits: it labels findings CONFIRMED /
REVERSED / NUANCED, it says "observational, not causal", and it flags the survivor bias in its smallest
cell. The questions below are the next step on that same ladder, not a correction. Every one of them I had
to ask about my own work first, and §2 answers it with a number I did not like.

## 1. Two paper findings + my methodology questions

### Finding #2 — "The Content Performance Curve" (paper p.7, marked CONFIRMED)

**The claim.** Health score by content age: 7 → 17 → 29 → 30 → **33 at 61–90 days** → 26 → 29 → 21 →
**14 at 271–365 days** → 25 at 365+. The paper reads this as a lifecycle: growth in the first 90 days, a
plateau, a decay cliff at 271–365 days, and a post-365 recovery concentrated in refreshed pages.

**Where the number comes from.** The outcome is the FlyRank health score, defined on p.5 as
impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts), measured over a rolling
90-day window at one point in time. The age axis is `days since content creation` in the same snapshot.

**The methodology question.** *Is the curve a lifecycle, or a cross-section of different pages?* Each age
bucket contains a different set of pages, not the same pages at different ages. The 61–90 bucket is
everything published two to three months before the snapshot; the 365+ bucket is only the pages that
survived a year *and* still had enough traffic to stay in the active-content subset. A between-page
snapshot cannot separate "pages get better at 61–90 days" from "the pages that happen to be 61–90 days old
in this snapshot are a different population". The paper already catches exactly this problem in its
smallest cell — it says the `365+ × 361+` quadrant is a survivor sample and must not be used as decay
proof (p.14). The constructive next step is to extend that same caveat to the whole curve: a
**within-page** design (follow the same pages as they cross age bands, or match cohorts on prior traffic)
would test lifecycle; the snapshot only describes composition. A second, smaller question: because health
score already contains impressions and position, part of any "peak" is indexation ramp rather than
content quality — a version of the curve on a raw outcome (weighted CTR, or clicks) would say more.

### Finding #4 — "The Freshness Multiplier" (paper p.9, marked CONFIRMED)

**The claim.** Pages aged 365+ that were **refreshed within 30 days** show a **3.2× health boost**
(10.7 → 34.5) and **57× more impressions** (71 → 4,039). Refresh timing is described as "one of the
strongest measured levers available".

**Where the number comes from.** The treatment is `days since last content update ≤ 30`; the outcome is
current health score and current impressions, again over the 90-day snapshot window. It is a comparison
between pages that were refreshed and pages that were not.

**The methodology question.** *What is the counterfactual — and does the 30-day treatment window overlap
the outcome window?* Two things would make me trust this number much more, and neither is a criticism of
the paper so much as the next increment of rigor:

1. **Selection.** Editors refresh pages they believe are worth refreshing — usually pages with proven
   demand. "Refreshed pages did better" then partly measures *which pages were chosen*, not what the
   refresh did. A matched comparison — old pages with similar prior impressions/position, refreshed vs
   untouched in the same window — is the design that answers the causal version of the question.
2. **Window alignment.** If a page is counted as refreshed when its last update falls within 30 days of
   the snapshot, the update timestamp shares a window with the 30-day impression measurement. The
   impressions that motivated the refresh and the impressions credited to it can be the same days.

There is also a base-size point the paper already makes well elsewhere: it refuses to headline the
283:1 freshness ratio because that bucket is 283 growing pages against **1** declining page (p.9). The
same standard applied to 71 → 4,039 impressions would make the flagship number stronger, not weaker.

**Why these two, and what they have to do with me.** Finding #2 is a **composition** problem and Finding
#4 is a **selection + window** problem. Those are the two ways my own Week-5 number could be wrong, and
§2 and §3 are where I find out. I could not re-run the paper's study, but I can run its design on data I
can inspect — which is the next cell.

In [1]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import duckdb
    from huggingface_hub import hf_hub_download
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"])
    import duckdb
    from huggingface_hub import hf_hub_download

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from threadpoolctl import threadpool_info, threadpool_limits

# Gradient boosting reduces histograms with OpenMP. With several threads the float reduction order
# differs between processes, which can flip a borderline pick and move a mean by ~0.01. One thread
# costs a few seconds and makes every number in this notebook reproducible run to run.
threadpool_limits(limits=1)
print("openmp threads:",
      [t["num_threads"] for t in threadpool_info() if t["internal_api"] == "openmp"])

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Paste your Hugging Face READ token: ")

REPO = "FlyRank/internship-warehouse"
FILES = {
    "march": "fact_content_daily_performance/month=2026-03/data_0.parquet",
    "april": "fact_content_daily_performance/month=2026-04/data_0.parquet",
    "dim_content": "dim_content.parquet",
}
local = {name: hf_hub_download(REPO, path, repo_type="dataset", token=HF_TOKEN)
         for name, path in FILES.items()}

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
for name, path in local.items():
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{Path(path).as_posix()}')")

KEYS = ["client_hash_id", "content_hash_id"]

# The feature query touches March and dim_content only. It is kept as a named string so §3 can audit it.
FEATURE_SQL = '''
WITH m AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_30d,
           SUM(gsc_clicks)      AS clicks_30d,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_30d,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_30d,
           SUM(CASE WHEN report_date <= DATE '2026-03-10' THEN gsc_impressions ELSE 0 END) AS imp_first10,
           SUM(CASE WHEN report_date >= DATE '2026-03-22' THEN gsc_impressions ELSE 0 END) AS imp_last10
    FROM march
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
)
SELECT m.*, c.content_type,
       DATEDIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
FROM m JOIN dim_content c USING (client_hash_id, content_hash_id)
ORDER BY client_hash_id, content_hash_id
'''

LABEL_SQL = '''
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr
FROM april WHERE gsc_data_available IS TRUE GROUP BY 1, 2
ORDER BY 1, 2
'''

march_volume = con.sql('''
SELECT COUNT(*) AS march_pages_all,
       SUM(CASE WHEN imp >= 100 THEN 1 ELSE 0 END) AS kept
FROM (SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp
      FROM march WHERE gsc_data_available IS TRUE GROUP BY 1, 2)
''').df()

pages = con.sql(FEATURE_SQL).df()
labels = con.sql(LABEL_SQL).df()

pages["ctr"] = 100.0 * pages.clicks_30d / pages.impressions_30d
pages["position_tier"] = pd.cut(pages.avg_position_30d, [0, 3, 10, 20, 50, 10000],
                                labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
pages["visible"] = (pages.avg_position_30d > 0) & (pages.avg_position_30d <= 20)
peer = pages[pages.visible].groupby("position_tier", observed=True).ctr.median()
pages["peer_ctr"] = pages.position_tier.map(peer)
pages["ctr_ratio"] = pages.ctr / pages.peer_ctr
pages["zero_clicks"] = (pages.clicks_30d == 0).astype(int)
pages["log_impressions"] = np.log1p(pages.impressions_30d)
pages["log_momentum"] = np.log((pages.imp_last10 + 1) / (pages.imp_first10 + 1))

gated = pages.visible & (pages.impressions_30d >= 500)
pts = np.select([gated & (pages.ctr < 0.5 * pages.peer_ctr),
                 gated & (pages.ctr < pages.peer_ctr)], [3, 1], default=0)
pages["baseline_score"] = 3 * pts + 2 * (pages.days_with_impressions_30d >= 25).astype(int)
pages["clicks_lost"] = (pages.peer_ctr / 100 * pages.impressions_30d - pages.clicks_30d).clip(lower=0)

df = pages.merge(labels, on=KEYS, how="inner")
df["declined"] = (df.imp_apr < 0.8 * df.impressions_30d).astype(int)
df = df[df.impressions_30d > 0].reset_index(drop=True)

FEATS_NUM = ["ctr_ratio", "zero_clicks", "log_impressions", "avg_position_30d",
             "days_with_impressions_30d", "content_age_days", "log_momentum"]
FEATS_CAT = ["content_type"]
FEATURES = FEATS_NUM + FEATS_CAT

print(f"March pages in the release: {int(march_volume.march_pages_all.iloc[0]):,}")
print(f"kept by the >=100 March impressions filter: {int(march_volume.kept.iloc[0]):,} "
      f"({march_volume.kept.iloc[0] / march_volume.march_pages_all.iloc[0]:.1%})")
print(f"frame after the April join: {len(df):,} pages | {df.client_hash_id.nunique()} clients "
      f"| base rate {df.declined.mean():.3f}")

# The paper's Finding #2 design, run on data I can inspect: mean outcome per age bucket.
# This is a cross-section, so every row of the table is a DIFFERENT set of pages.
bins = [-1, 30, 60, 90, 180, 270, 365, 100000]
labels_age = ["0-30", "31-60", "61-90", "91-180", "181-270", "271-365", "365+"]
df["age_band"] = pd.cut(df.content_age_days, bins, labels=labels_age)
cross = df.groupby("age_band", observed=True).agg(
    pages=("declined", "size"),
    decline_rate=("declined", "mean"),
    mean_impressions=("impressions_30d", "mean"),
    mean_ctr=("ctr", "mean"),
    mean_position=("avg_position_30d", "mean"),
)
cross["page_share"] = (cross.pages / cross.pages.sum()).round(3)
print("\nthe same snapshot design on my frame — age bucket vs outcome:")
print(cross.round(3).to_string())
print("\neach row is a different population, not the same pages followed forward:")
print(f"  the biggest bucket holds {cross.pages.max():,} pages, the smallest {cross.pages.min():,} — "
      f"a {cross.pages.max() / max(cross.pages.min(), 1):.0f}x difference in what the bucket is averaging over.")
print(f"  the 365+ bucket only contains pages that survived to be measured in March at all: "
      f"{cross.loc['365+', 'pages']:,} of them, with mean position {cross.loc['365+', 'mean_position']:.1f}.")

openmp threads: [1]


March pages in the release: 176,738
kept by the >=100 March impressions filter: 101,441 (57.4%)
frame after the April join: 100,893 pages | 43 clients | base rate 0.515

the same snapshot design on my frame — age bucket vs outcome:
          pages  decline_rate  mean_impressions  mean_ctr  mean_position  page_share
age_band                                                                            
0-30       8560         0.271          1166.223     0.427         12.178       0.085
31-60     12514         0.434          2428.472     0.372         10.424       0.124
61-90     11996         0.515          3250.856     0.211         13.320       0.119
91-180    16082         0.618          3511.915     0.223         15.205       0.159
181-270   28161         0.614          3066.038     0.247         15.252       0.279
271-365    9266         0.393          2468.199     0.218         17.722       0.092
365+      14314         0.499          2342.761     0.212         17.844       0.142

ea

**What that table shows.** Run the paper's Finding #2 design on my own snapshot and it produces exactly the
kind of table that invites a lifecycle story — but it is not one. The buckets differ **3×** in size
(8,560 to 28,161 pages), the movement is not monotone, and the 365+ bucket contains only pages that
survived long enough to still be measured in March, at a mean position of 17.8 against 12.2 for the 0–30
day bucket. Different pages, different positions, one snapshot. That is precisely the question I asked of
the paper, and I cannot answer it from a one-month panel either — so the honest statement about my own
work is: **this design describes composition, not lifecycle.** The same sentence I would want the paper to
be able to say.

That matters because my model is built on the same kind of snapshot. Everything from here is about the
other failure mode: a validation design that quietly lets the model see the answer.

> **Frame note.** Seven of the eight features are March aggregates or static `dim_content` metadata, so they
> are known on 2026-03-31 by construction. The eighth, `log_momentum`, uses only **2026-03-01..03-10** and
> **2026-03-22..03-31** — both windows close before the decision moment. The label is April only. §3 proves
> this rather than asserting it.

## 2. My model under an honest split (before/after)

ML-08 shipped Histogram Gradient Boosting and reported P@50 **0.820** on grouped holdouts. Before
defending that number I want to know what the *wrong* split would have said — because if a random page
split had produced a much prettier figure, that is the figure I would have been tempted to report.

**The two designs, identical in every other respect** (same frame, same 8 features, same label, same
model, same hyperparameters, same 80/20 ratio, five seeds each):

| | before — random page split | after — grouped client split |
|---|---|---|
| Unit held out | rows | `client_hash_id` |
| Sibling pages of a test page in train | yes, almost always | never |
| Answers | "can it score a page from a site it has seen?" | "can it score a page from a client it has never seen?" |

The second question is the one deployment asks. A new client arrives as a whole site: same templates, same
CMS, same audience, same internal linking, same click behaviour. If the model learns the *site*, a random
split will reward it for memorising a client it will never meet again.

To make the mechanism visible rather than asserted, the cell below also prints the **intraclass
correlation** of the label across clients — how much of the yes/no outcome is a client-level property —
and the share of test rows whose client also appears in training.

In [2]:
from sklearn.linear_model import LogisticRegression

SEEDS = [42, 7, 2024, 101, 555]
TEST_SIZE = 0.2


def make_hgb(num, cat=FEATS_CAT):
    return Pipeline([
        ("pre", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), num),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
        ])),
        ("clf", HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, max_depth=4,
                                               min_samples_leaf=40, random_state=42)),
    ])


def p_at_k(y, order, k):
    return float(np.asarray(y)[order[:k]].mean())


def baseline_positions(frame):
    reset = frame.reset_index(drop=True)
    return reset.sort_values(["baseline_score", "clicks_lost", "impressions_30d"],
                             ascending=False).index.values


def score_metrics(y, score, order):
    return ({f"P@{k}": round(p_at_k(y, order, k), 3) for k in (10, 20, 50, 100)}
            | {"AUC": round(roc_auc_score(y, score), 3),
               "PR-AUC": round(average_precision_score(y, score), 3)})


def grouped_split(frame, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
    tri, tei = next(gss.split(frame, groups=frame.client_hash_id))
    assert not set(frame.iloc[tri].client_hash_id) & set(frame.iloc[tei].client_hash_id)
    return tri, tei


def random_split(frame, seed):
    tri, tei = train_test_split(np.arange(len(frame)), test_size=TEST_SIZE,
                                random_state=seed, stratify=frame.declined)
    return tri, tei


# ---- intraclass correlation of the label across clients (one-way ICC(1), binary outcome)
grp = df.groupby("client_hash_id").declined
n_i = grp.size().values.astype(float)
mu_i = grp.mean().values
grand = df.declined.mean()
k, N = len(n_i), n_i.sum()
ms_between = (n_i * (mu_i - grand) ** 2).sum() / (k - 1)
ms_within = (n_i * mu_i * (1 - mu_i)).sum() / (N - k)
icc = (ms_between - ms_within) / (ms_between + (N - k) / (k - 1) * ms_within)

print(f"label is partly a client-level property: ICC(1) = {icc:.3f} across {k} clients")
print(f"  client decline rates: min {mu_i.min():.3f} | median {np.median(mu_i):.3f} | max {mu_i.max():.3f}")
print(f"  a random page split lets the model learn that client rate from sibling pages; a grouped split cannot.\n")

runs = []
for seed in SEEDS:
    for design, splitter in (("before: random page", random_split),
                             ("after: grouped client", grouped_split)):
        tri, tei = splitter(df, seed)
        tr, te = df.iloc[tri], df.iloc[tei]
        sibling_share = float(te.client_hash_id.isin(set(tr.client_hash_id)).mean())

        model = make_hgb(FEATS_NUM).fit(tr[FEATURES], tr.declined)
        s = model.predict_proba(te[FEATURES])[:, 1]
        row = {"design": design, "seed": seed, "test_rows": len(te),
               "sibling_share": round(sibling_share, 3),
               **score_metrics(te.declined.values, s, np.argsort(-s))}

        # the frozen rule on the identical test rows: a control, because a fixed rule cannot memorise a client
        rule = score_metrics(te.declined.values, te.baseline_score.values, baseline_positions(te))
        row["rule_P@50"] = rule["P@50"]

        # the ablation, inside the honest design
        no_mom_cols = [f for f in FEATURES if f != "log_momentum"]
        no_mom = make_hgb([f for f in FEATS_NUM if f != "log_momentum"]).fit(tr[no_mom_cols], tr.declined)
        s_nm = no_mom.predict_proba(te[no_mom_cols])[:, 1]
        row["P@50_no_momentum"] = score_metrics(te.declined.values, s_nm, np.argsort(-s_nm))["P@50"]
        runs.append(row)

runs_df = pd.DataFrame(runs)
by_design = runs_df.groupby("design")[["P@10", "P@20", "P@50", "P@100", "AUC", "PR-AUC",
                                       "rule_P@50", "P@50_no_momentum"]].mean()

print("five seeds per design — mean across seeds, identical frame/label/model:")
print(by_design.round(3).to_string())
print("\nper-seed P@50 (the spread is why the headline is a mean):")
print(runs_df.pivot(index="seed", columns="design", values="P@50").round(3).to_string())
print("\ntest rows whose client also appears in training:")
print(runs_df.groupby("design").sibling_share.agg(["mean", "min", "max"]).round(3).to_string())

rand = runs_df[runs_df.design == "before: random page"]
group = runs_df[runs_df.design == "after: grouped client"]
print("\nwhat the wrong split would have let me claim:")
print(f"  P@50  {rand['P@50'].mean():.3f} (random) vs {group['P@50'].mean():.3f} (grouped) "
      f"— inflated by {rand['P@50'].mean() - group['P@50'].mean():+.3f}")
print(f"  AUC   {rand.AUC.mean():.3f} (random) vs {group.AUC.mean():.3f} (grouped) "
      f"— inflated by {rand.AUC.mean() - group.AUC.mean():+.3f}")
print(f"  the frozen rule moves much less ({rand['rule_P@50'].mean():.3f} vs "
      f"{group['rule_P@50'].mean():.3f}): "
      "a fixed rule has nothing to memorise, so most of the gap is the learner's, not the split's.")

label is partly a client-level property: ICC(1) = 0.133 across 43 clients
  client decline rates: min 0.000 | median 0.472 | max 1.000
  a random page split lets the model learn that client rate from sibling pages; a grouped split cannot.



five seeds per design — mean across seeds, identical frame/label/model:
                       P@10  P@20   P@50  P@100    AUC  PR-AUC  rule_P@50  P@50_no_momentum
design                                                                                     
after: grouped client  0.84  0.79  0.820  0.806  0.643   0.591      0.540             0.516
before: random page    1.00  1.00  0.992  0.988  0.791   0.798      0.664             0.976

per-seed P@50 (the spread is why the headline is a mean):
design  after: grouped client  before: random page
seed                                              
7                        0.88                 1.00
42                       0.62                 1.00
101                      0.94                 0.98
555                      0.80                 1.00
2024                     0.86                 0.98

test rows whose client also appears in training:
                       mean  min  max
design                               
after: grouped cli

In [3]:
# Where the inflation comes from: the random-split head concentrates on a couple of familiar clients.
tri, tei = random_split(df, 42)
tr, te = df.iloc[tri], df.iloc[tei]
s_rand = make_hgb(FEATS_NUM).fit(tr[FEATURES], tr.declined).predict_proba(te[FEATURES])[:, 1]
top_rand = te.iloc[np.argsort(-s_rand)[:50]]

tri, tei = grouped_split(df, 42)
tr, te = df.iloc[tri], df.iloc[tei]
s_grp = make_hgb(FEATS_NUM).fit(tr[FEATURES], tr.declined).predict_proba(te[FEATURES])[:, 1]
top_grp = te.iloc[np.argsort(-s_grp)[:50]]

print("top 50 of the queue, seed 42:")
print(f"  random page split   : {top_rand.client_hash_id.nunique()} distinct clients | "
      f"{int(top_rand.declined.sum())}/50 declined | largest client supplies "
      f"{int(top_rand.client_hash_id.value_counts().iloc[0])}/50 picks")
print(f"  grouped client split: {top_grp.client_hash_id.nunique()} distinct clients | "
      f"{int(top_grp.declined.sum())}/50 declined | largest client supplies "
      f"{int(top_grp.client_hash_id.value_counts().iloc[0])}/50 picks")
print("\nthe random split's head is not a better model; it is the model correctly guessing which sites")
print("it already met in training. On a new client that information does not exist.")

top 50 of the queue, seed 42:
  random page split   : 3 distinct clients | 50/50 declined | largest client supplies 33/50 picks
  grouped client split: 2 distinct clients | 31/50 declined | largest client supplies 46/50 picks

the random split's head is not a better model; it is the model correctly guessing which sites
it already met in training. On a new client that information does not exist.


### What the before/after says

The grouped split is not a cosmetic correction. On the same model, the same features and the same label:

- **P@50: 0.992 → 0.820.** The random split is near-perfect at the head of the queue, and it is near-perfect
  because 100% of its test rows have a sibling in training. Its top 50 comes from three clients, one of which
  supplies 33 of the 50 picks — and all 50 decline.
- **Concentration is not the discriminator; knowing the client is.** The grouped head is *also* concentrated
  (46 of its 50 picks come from one client) but only 31 of them decline. Same shape, different reason: one
  model is recalling a site it has seen, the other is guessing inside a site it has not.
- **The ablation proves it.** Under the random split, dropping `log_momentum` still leaves P@50 at **0.976**.
  Under the grouped split the same ablation drops to **0.516**. The random split's performance was never
  about the feature — it was about the siblings.
- **AUC: 0.791 → 0.643.** The inflation is largest exactly where the queue is read.
- **The frozen rule barely moves** (0.664 → 0.540). A fixed rule has no client memory to exploit, which is
  a useful control: most of the gap belongs to the learner, not to the split's sampling noise.

So the number I stand behind is **0.820**, and the number I would have reported if I had let rows split
randomly is **0.992** — a figure that would have been cited as a near-perfect queue and would have
collapsed on the first new client. Reporting the second one would have been the same error as publishing
a cross-sectional age curve as a lifecycle: the design, not the math, does the flattering.

## 3. Leakage audit

The Week-3 hunt, re-run on the final feature set. Four attacks, each with a number:

1. **Feature provenance.** Every feature must trace to a column and a date window that closes before
   2026-03-31. The feature SQL is a named string, so "does it touch April?" is a check, not a promise.
2. **Label injection.** Force the label's own ingredient into the model (`imp_apr`), then a near-label
   ratio, and watch what happens. The confession is a jump to ~1.0 and a clean return when removed.
3. **Ablation.** Drop `log_momentum` and see whether the model depends on it the way a leak-dependent
   model would.
4. **Population selection.** Who never makes it into the frame, and does that change what the metric means?

In [4]:
# ---- 1. provenance: the feature query must not mention April, and every window must close before 04-01
MANIFEST = pd.DataFrame([
    {"feature": "ctr_ratio", "source": "March gsc_clicks / gsc_impressions, divided by tier peer median",
     "window": "2026-03-01..2026-03-31", "known_at": "2026-03-31"},
    {"feature": "zero_clicks", "source": "March gsc_clicks", "window": "2026-03-01..2026-03-31",
     "known_at": "2026-03-31"},
    {"feature": "log_impressions", "source": "March gsc_impressions", "window": "2026-03-01..2026-03-31",
     "known_at": "2026-03-31"},
    {"feature": "avg_position_30d", "source": "March gsc_avg_position (>0 rows only)",
     "window": "2026-03-01..2026-03-31", "known_at": "2026-03-31"},
    {"feature": "days_with_impressions_30d", "source": "March report_date with gsc_impressions > 0",
     "window": "2026-03-01..2026-03-31", "known_at": "2026-03-31"},
    {"feature": "content_age_days", "source": "dim_content.content_created_date",
     "window": "static metadata", "known_at": "always"},
    {"feature": "log_momentum", "source": "March gsc_impressions, first ten vs last ten days",
     "window": "2026-03-01..03-10 and 2026-03-22..03-31", "known_at": "2026-03-31"},
    {"feature": "content_type", "source": "dim_content.content_type", "window": "static metadata",
     "known_at": "always"},
])

DECISION_MOMENT = pd.Timestamp("2026-03-31")
LB_WINDOW_START = pd.Timestamp("2026-04-01")

print("feature provenance:")
print(MANIFEST.to_string(index=False))

latest_used = max(pd.Timestamp(d) for w in MANIFEST.window for d in
                  re.findall(r"\d{4}-\d{2}-\d{2}", w))
print("\nstructural checks:")
print(f"  the feature query references the April table: {'april' in FEATURE_SQL.lower()}")
print(f"  the label query references the April table:   {'april' in LABEL_SQL.lower()}")
print(f"  latest date used by any feature: {latest_used.date()} | decision moment: {DECISION_MOMENT.date()} "
      f"-> {latest_used <= DECISION_MOMENT}")
print(f"  label window starts {LB_WINDOW_START.date()}, after the decision moment: "
      f"{LB_WINDOW_START > DECISION_MOMENT}")
assert latest_used <= DECISION_MOMENT
assert "april" not in FEATURE_SQL.lower()

# product / workflow columns that would smuggle in a decision or a target
PRODUCT_COLS = {"health_score", "optimization_flag", "fix_ctr", "zombie", "trend_direction",
                "trend_pct", "is_declining_label", "priority", "action"}
present = sorted(PRODUCT_COLS & set(df.columns))
print(f"  FlyRank product/flag columns present in the frame: {present if present else 'none'}")

FORBIDDEN = {"imp_apr", "declined", "baseline_score", "ctr_points", "clicks_lost"}
print(f"  label or rule columns inside FEATURES: {sorted(FORBIDDEN & set(FEATURES)) or 'none'}")
assert not (set(FEATURES) & FORBIDDEN)

feature provenance:
                  feature                                                          source                                  window   known_at
                ctr_ratio March gsc_clicks / gsc_impressions, divided by tier peer median                  2026-03-01..2026-03-31 2026-03-31
              zero_clicks                                                March gsc_clicks                  2026-03-01..2026-03-31 2026-03-31
          log_impressions                                           March gsc_impressions                  2026-03-01..2026-03-31 2026-03-31
         avg_position_30d                           March gsc_avg_position (>0 rows only)                  2026-03-01..2026-03-31 2026-03-31
days_with_impressions_30d                      March report_date with gsc_impressions > 0                  2026-03-01..2026-03-31 2026-03-31
         content_age_days                                dim_content.content_created_date                         static metadata     

In [5]:
# ---- 2 and 3. injection and ablation on the honest split (grouped, seed 42)
tri, tei = grouped_split(df, 42)
tr, te = df.iloc[tri].copy(), df.iloc[tei].copy()
tr["april_ratio"] = tr.imp_apr / tr.impressions_30d
te["april_ratio"] = te.imp_apr / te.impressions_30d

VARIANTS = {
    "shipped features": (FEATS_NUM, FEATS_CAT),
    "+ imp_apr (the label's own ingredient)": (FEATS_NUM + ["imp_apr"], FEATS_CAT),
    "+ april_ratio (near-label)": (FEATS_NUM + ["april_ratio"], FEATS_CAT),
    "- log_momentum (ablation)": ([f for f in FEATS_NUM if f != "log_momentum"], FEATS_CAT),
}

leak_rows = []
for name, (num, cat) in VARIANTS.items():
    model = make_hgb(num, cat).fit(tr[num + cat], tr.declined)
    s = model.predict_proba(te[num + cat])[:, 1]
    leak_rows.append({"variant": name, **score_metrics(te.declined.values, s, np.argsort(-s))})

leak_df = pd.DataFrame(leak_rows).set_index("variant")
print("injection and ablation on the grouped seed-42 holdout:")
print(leak_df.to_string())

clean_auc = leak_df.loc["shipped features", "AUC"]
inject_auc = leak_df.loc["+ imp_apr (the label's own ingredient)", "AUC"]
print(f"\nAUC goes {clean_auc:.3f} -> {inject_auc:.3f} when the label's ingredient is added "
      f"({inject_auc - clean_auc:+.3f}); the same fitted model on the shipped columns alone scores "
      f"{clean_auc:.3f}, so nothing of the leak stays behind.")

top_importance = leak_df.loc["- log_momentum (ablation)"]
print(f"\nthe ablation is the control: without one honest feature the model scores "
      f"{leak_df.loc['shipped features', 'AUC'] - top_importance.AUC:.3f} AUC and "
      f"{leak_df.loc['shipped features', 'P@50'] - top_importance['P@50']:.3f} P@50 lower.")
print("  a label leak does not degrade gracefully — it saturates at ~1.0. This model degrades to the")
print("  rule's level instead, which is what a real but imperfect signal looks like.")

injection and ablation on the grouped seed-42 holdout:
                                        P@10  P@20  P@50  P@100    AUC  PR-AUC
variant                                                                       
shipped features                         0.7   0.6  0.62   0.64  0.617   0.524
+ imp_apr (the label's own ingredient)   1.0   1.0  1.00   1.00  0.999   0.999
+ april_ratio (near-label)               1.0   1.0  1.00   1.00  1.000   1.000
- log_momentum (ablation)                0.5   0.5  0.54   0.53  0.554   0.456

AUC goes 0.617 -> 0.999 when the label's ingredient is added (+0.382); the same fitted model on the shipped columns alone scores 0.617, so nothing of the leak stays behind.

the ablation is the control: without one honest feature the model scores 0.063 AUC and 0.080 P@50 lower.
  a label leak does not degrade gracefully — it saturates at ~1.0. This model degrades to the
  rule's level instead, which is what a real but imperfect signal looks like.


In [6]:
# ---- 4. population selection: who never makes it into the frame, and does it matter?
keys_df = df.set_index(KEYS).index
excluded = pages[~pages.set_index(KEYS).index.isin(keys_df)]
no_april_clients = set(pages.client_hash_id) - set(df.client_hash_id)

print("who is missing, and why:")
print(f"  March pages reaching the 100-impression filter : {len(pages):,}")
print(f"  ... with an April row, i.e. the frame         : {len(df):,}")
print(f"  ... dropped by the April join                 : {len(excluded):,} pages from "
      f"{excluded.client_hash_id.nunique()} clients")
print(f"  clients with no April rows at all             : {len(no_april_clients)}")

if len(excluded):
    print("\nprofile of the 548 dropped pages vs the frame they left:")
    prof = pd.DataFrame({
        "dropped": [len(excluded), excluded.impressions_30d.mean(), excluded.impressions_30d.median(),
                    excluded.clicks_30d.mean(), excluded.visible.mean(),
                    excluded.days_with_impressions_30d.median()],
        "kept": [len(df), df.impressions_30d.mean(), df.impressions_30d.median(),
                 df.clicks_30d.mean(), df.visible.mean(), df.days_with_impressions_30d.median()],
    }, index=["pages", "mean March impressions", "median March impressions", "mean March clicks",
              "share visible (pos 1-20)", "median days with impressions"])
    print(prof.round(3).to_string())

print("\nwhat the selection rule means for the metric:")
print(f"  the frame is {len(df) / int(march_volume.march_pages_all.iloc[0]):.1%} of all March pages: "
      "57.4% pass the 100-impression filter and 548 more lack an April row.")
print("  so every number in this notebook is about pages that already had measurable search demand.")
print("  a queue built on it cannot rank the long tail, and the 80%-of-March label is noisier at low volume.")
print(f"  the frozen baseline's own scoring gate is 500 impressions, so it only ever scored "
      f"{int((df.visible & (df.impressions_30d >= 500)).sum()):,} of these {len(df):,} pages.")

who is missing, and why:
  March pages reaching the 100-impression filter : 101,441
  ... with an April row, i.e. the frame         : 100,893
  ... dropped by the April join                 : 548 pages from 30 clients
  clients with no April rows at all             : 1

profile of the 548 dropped pages vs the frame they left:
                              dropped        kept
pages                         548.000  100893.000
mean March impressions        365.230    2761.301
median March impressions      188.500     794.000
mean March clicks               0.648       8.078
share visible (pos 1-20)        0.790       0.755
median days with impressions   20.000      31.000

what the selection rule means for the metric:
  the frame is 57.1% of all March pages: 57.4% pass the 100-impression filter and 548 more lack an April row.
  so every number in this notebook is about pages that already had measurable search demand.
  a queue built on it cannot rank the long tail, and the 80%-of-March la

### Leakage verdict

**Nothing leaked, and the audit can say why.** The feature query never names the April table
(`'april' in FEATURE_SQL` is `False`, asserted); the label query is separate and joins after the split;
no product or workflow flag exists in the frame at all; and every feature window closes on 2026-03-31.

The injection is the part that makes it believable rather than merely asserted: adding `imp_apr` takes AUC
from **0.617 → 0.999** and P@50 from **0.620 → 1.000**; the near-label ratio does the same. That is what a
real leak looks like — it does not add a little, it saturates. Drop those columns and the same features
score identically (0.617), while the honest ablation degrades gently to the rule's level instead.

**The one real finding is about population, not leakage.** The frame keeps 57% of March pages — only those
with ≥100 impressions — so every claim in this notebook is a claim about pages that already had measurable
demand. That is the same limitation the paper discloses about its active-content subset, and I should be
as explicit as the paper is: **this queue cannot rank the long tail, and it should never be described as
covering the portfolio.** A second, smaller effect: the 548 pages dropped by the April join are small and
mostly visible (median 189 March impressions), so they do not move the rates much, but they are gone.

Also worth stating plainly: the label compares **April with the whole of March**, so `log_momentum` — which
compares the last ten days of March with the first ten — is partly *mechanically* related to it. That is not
leakage (the feature is a fact about the past, and it demonstrably helps on unseen clients), but it is the
reason the model's head is populated by pages that were already fading. It is an association inside one
month pair, not a law.

## 4. Claim rewrite

ML-08 ended with sentences I would now write differently. The audit gives me the numbers to fix them, and
the fixes are not decorative — two of the four change what the sentence is allowed to mean.

The rule I am applying: **observed** for something printed in a table, **measured** for a number with a
stated scope and spread, **directional** for a sign I believe but cannot size, **decision-support** for what
the output is for. Nothing here is causal, because nothing here was randomised.

In [7]:
group = runs_df[runs_df.design == "after: grouped client"]
rand = runs_df[runs_df.design == "before: random page"]
g_p50, g_p10, g_auc = group["P@50"].mean(), group["P@10"].mean(), group.AUC.mean()
g_p50_min, g_p50_max = group["P@50"].min(), group["P@50"].max()
r_p50 = rand["P@50"].mean()
nomom = group["P@50_no_momentum"].mean()
rule = group["rule_P@50"].mean()

claims = pd.DataFrame([
    {
        "claim as written in ML-08": "The ensemble ships for the queue head.",
        "what the audit measured": (f"grouped holdouts: mean P@50 {g_p50:.3f} "
                                    f"(range {g_p50_min:.2f}-{g_p50_max:.2f}), mean P@10 {g_p10:.3f}, "
                                    f"AUC {g_auc:.3f}, five draws over nine held-out clients"),
        "safe rewrite": ("Decision support only: on pages from clients the model had not seen, it measured a "
                         "higher mean precision at the head of the queue than the frozen rule. The spread "
                         "across draws is wide and nine clients per draw is a small sample."),
    },
    {
        "claim as written in ML-08": "74% of the lift is the feature, not the model class.",
        "what the audit measured": (f"removing log_momentum drops P@50 to {nomom:.3f}, at or below the "
                                    f"rule's {rule:.3f}; the percentage depended on which model carried the "
                                    "decomposition"),
        "safe rewrite": ("Directional: most of the measured lift is associated with one observable feature "
                         "(within-March momentum). The ablation is the evidence; the exact percentage is "
                         "an artefact of the decomposition, so I no longer quote it."),
    },
    {
        "claim as written in ML-08": "P@50 0.820 beats the baseline's 0.740.",
        "what the audit measured": (f"the two numbers come from different universes: {g_p50:.3f} is a "
                                    f"nine-client mean, 0.740 was measured on all 43 clients"),
        "safe rewrite": ("On identical held-out rows the rule scored a mean P@50 of "
                         f"{rule:.3f} and the model {g_p50:.3f}. The 0.740 figure is a different "
                         "evaluation universe and is not a comparison."),
    },
    {
        "claim as written in ML-08": "Momentum predicts which pages will decline.",
        "what the audit measured": (f"removing the feature lowers mean P@50 by "
                                    f"{group['P@50'].mean() - nomom:.3f} on held-out clients; the label "
                                    "compares April with all of March, so the two overlap in construction"),
        "safe rewrite": ("Observed association: in this one month pair, pages whose March impressions were "
                         "already falling were more likely to fall below 80% of March in April. This is not "
                         "evidence that the trend causes the decline, and one month pair is one sample."),
    },
])

for i, row in claims.iterrows():
    print(f"\n[{i + 1}] CLAIM AS WRITTEN")
    print(f"    {row['claim as written in ML-08']}")
    print("    WHAT THE AUDIT MEASURED")
    print(f"    {row['what the audit measured']}")
    print("    SAFE REWRITE")
    print(f"    {row['safe rewrite']}")

print("\nthe number I will quote, and the number I will not:")
print(f"  quoted     : P@50 {g_p50:.3f} measured on grouped client holdouts, "
      f"{g_p50_min:.2f}-{g_p50_max:.2f} across five draws")
print(f"  not quoted : P@50 {r_p50:.3f} from the random page split — a split that puts siblings on both sides")


[1] CLAIM AS WRITTEN
    The ensemble ships for the queue head.
    WHAT THE AUDIT MEASURED
    grouped holdouts: mean P@50 0.820 (range 0.62-0.94), mean P@10 0.840, AUC 0.643, five draws over nine held-out clients
    SAFE REWRITE
    Decision support only: on pages from clients the model had not seen, it measured a higher mean precision at the head of the queue than the frozen rule. The spread across draws is wide and nine clients per draw is a small sample.

[2] CLAIM AS WRITTEN
    74% of the lift is the feature, not the model class.
    WHAT THE AUDIT MEASURED
    removing log_momentum drops P@50 to 0.516, at or below the rule's 0.540; the percentage depended on which model carried the decomposition
    SAFE REWRITE
    Directional: most of the measured lift is associated with one observable feature (within-March momentum). The ablation is the evidence; the exact percentage is an artefact of the decomposition, so I no longer quote it.

[3] CLAIM AS WRITTEN
    P@50 0.820 beats th

### The sentence I am actually willing to defend

> On pages from nine clients the model had not seen, it **measured** a higher mean precision at the head of
> the queue than the frozen rule on the identical rows and label — P@50 0.820 (0.62–0.94 across five
> draws) against 0.540. Most of that gap is associated with one observable feature, within-March momentum:
> removing it returns the model to the rule's level. This is **decision support** for which pages to look at
> first. It is not a claim that fixing a page recovers its traffic, not a claim about why any page declined,
> and not a claim about the 43% of March pages that never earned 100 impressions.

That is a smaller sentence than ML-08's, and it is the one the evidence actually carries.

## 5. Self-check

- [x] **Two paper findings, each with a methodology question, in a constructive tone.** §1 takes Finding #2
      (the age/health curve) and Finding #4 (the freshness multiplier), names where each number comes from,
      and asks the question the design cannot answer — composition for #2, selection and window overlap for
      #4 — while acknowledging the caveats the paper already discloses.
- [x] **My own model re-run under an honest split, with before and after.** §2 runs the identical model,
      features, label and hyperparameters under a random page split and a grouped client split, five seeds
      each: P@50 0.992 → 0.820, AUC 0.791 → 0.643. The frozen rule is scored as a control because it cannot
      memorise a client, and it moves far less. §2 also prints the label's ICC across clients and the share
      of test rows with a sibling in training, so the mechanism is measured, not asserted.
- [x] **A leakage audit.** §3 checks provenance against a named feature query, asserts that query never
      touches April, lists product/flag columns (there are none in the frame), injects `imp_apr` and a
      near-label ratio to show the saturation signature (AUC → 0.999), removes them, and ablates
      `log_momentum` to show the difference between a leak and an imperfect signal.
- [x] **Error examples.** §2 reports the concentration of the inflated split's head (three clients, one of
      them supplying 33 of 50 picks) and the per-seed spread, and §3 reports the population that never
      enters the frame — 43% of March pages, plus 548 dropped by the April join.
- [x] **Claims use public-safe language.** §4 rewrites four ML-08 sentences and labels each with observed /
      measured / directional / decision-support. No causal claim appears anywhere in this notebook.
- [x] **No client names, URLs, or raw queries.** Only pseudonymous hash ids appear, and no page text is
      printed.
- [x] **Runs top to bottom with no errors** — executed with outputs visible before committing.

**What this notebook changes about my capstone.** Two things. The number I report is the grouped-split
number with its spread, never the random-split one. And the claim I make is decision support for ordering a
queue on pages that already have measurable demand — with the population limit stated in the same breath,
the way the paper states its own.

In [8]:
import json


def repo_root(start):
    for cand in [start, *start.parents]:
        if (cand / "work" / "notebooks").is_dir():
            return cand
    return start


OUT = repo_root(Path.cwd()) / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

receipt = {
    "notebook": "w06_validation_audit.ipynb",
    "lane": "Lane 4 - CTR / Engagement Opportunity Scoring",
    "decision_moment": "2026-03-31",
    "frame": {"march_pages_total": int(march_volume.march_pages_all.iloc[0]),
              "kept_by_100_impression_filter": int(march_volume.kept.iloc[0]),
              "frame_pages": int(len(df)), "clients": int(df.client_hash_id.nunique()),
              "base_rate": round(float(df.declined.mean()), 4),
              "dropped_by_april_join": int(len(excluded)),
              "clients_without_april": len(no_april_clients)},
    "split_comparison": {
        "design_before": "random page split, stratified on the label, 80/20",
        "design_after": "grouped by client_hash_id, GroupShuffleSplit, test 20% of clients, 80/20",
        "seeds": SEEDS,
        "sibling_share_in_test": {
            "random_page": round(float(rand.sibling_share.mean()), 3),
            "grouped_client": round(float(group.sibling_share.mean()), 3)},
        "label_icc_across_clients": round(float(icc), 4),
        "before": {m: round(float(rand[m].mean()), 3)
                   for m in ("P@10", "P@20", "P@50", "P@100", "AUC", "PR-AUC", "rule_P@50")},
        "after": {m: round(float(group[m].mean()), 3)
                  for m in ("P@10", "P@20", "P@50", "P@100", "AUC", "PR-AUC", "rule_P@50")},
        "after_p_at_50_range": [round(float(group["P@50"].min()), 3),
                                round(float(group["P@50"].max()), 3)],
        "ablation_no_momentum_p_at_50_after": round(float(nomom), 3),
    },
    "leakage": {
        "feature_query_touches_april": "april" in FEATURE_SQL.lower(),
        "product_flag_columns_present": present,
        "injection": {k: {"AUC": float(v.AUC), "P@50": float(v["P@50"])}
                      for k, v in leak_df.iterrows()},
        "verdict": "no label or future-window leak found; injection saturates near 1.0 as expected",
    },
    "paper_audit": {
        "finding_2": {"title": "The Content Performance Curve (p.7)",
                      "outcome": "FlyRank health score = 30 impressions + 30 position + 20 CTR + 20 scroll",
                      "question": "cross-section of different pages, not a within-page lifecycle; "
                                  "365+ bucket is a survivor sample"},
        "finding_4": {"title": "The Freshness Multiplier (p.9)",
                      "outcome": "current health score and impressions, treatment = refreshed within 30 days",
                      "question": "no matched control for selection into refresh; 30-day treatment window "
                                  "overlaps the 30-day measurement window"},
    },
    "limitations": [
        "the frame is 57% of March pages (>=100 impressions); the long tail is out of scope",
        "nine clients per held-out draw, so the mean across five seeds is the headline, not one draw",
        "the label compares April with all of March, so within-March momentum is partly mechanical",
        "the population is one month pair; nothing here is causal or repeats over time",
    ],
}
(OUT / "w06_validation_audit.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
print("wrote work/outputs/w06_validation_audit.json")
print(json.dumps(receipt["split_comparison"]["before"], indent=2))
print(json.dumps(receipt["split_comparison"]["after"], indent=2))

wrote work/outputs/w06_validation_audit.json
{
  "P@10": 1.0,
  "P@20": 1.0,
  "P@50": 0.992,
  "P@100": 0.988,
  "AUC": 0.791,
  "PR-AUC": 0.798,
  "rule_P@50": 0.664
}
{
  "P@10": 0.84,
  "P@20": 0.79,
  "P@50": 0.82,
  "P@100": 0.806,
  "AUC": 0.643,
  "PR-AUC": 0.591,
  "rule_P@50": 0.54
}
